# LeetCode 868 — Binary Gap bằng Python

**Link đề:** https://leetcode.com/problems/binary-gap/
**Độ khó:** Easy · **Chủ đề:** Bit Manipulation

```python
class Solution:
    def binaryGap(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Đọc biểu diễn **nhị phân** của số nguyên: `bin(n)`, vị trí bit (bit 0 là bit **phải nhất**).
2. Hiểu "hai bit 1 **kề nhau**" (chỉ có 0 ở giữa) và khoảng cách = hiệu vị trí.
3. Đi từ cách chuỗi (`bin` + danh sách vị trí) → quét chuỗi một lượt → **thao tác bit thuần** với `n & 1`, `n >>= 1`.
4. Nắm kỹ thuật "**nhớ vị trí phần tử thỏa điều kiện gần nhất**" (`last`) — giống nhiều bài mảng khác.
5. Trace, chứng minh, test (lũy thừa của 2, toàn bit 1, số lớn) và mở rộng sang các mẹo bit `n & (n - 1)`, `n & -n`.

## 1. Đề bài và ví dụ

Cho số nguyên dương `n`. Tìm **khoảng cách lớn nhất** giữa hai bit `1` **kề nhau** trong biểu diễn nhị phân của `n`. Nếu không có hai bit 1 kề nhau, trả `0`.

- Hai bit 1 **kề nhau** nếu giữa chúng chỉ có bit 0 (có thể không có bit 0 nào).
- Khoảng cách = hiệu tuyệt đối vị trí hai bit. Ví dụ hai bit 1 trong `"1001"` cách nhau `3`.

| n | nhị phân | các cặp 1 kề nhau (khoảng cách) | Kết quả |
|---|---|---|---|
| 22 | `10110` | `1_011_0`→ 2; `10_11_0` → 1 | 2 |
| 8 | `1000` | không có | 0 |
| 5 | `101` | 2 | 2 |

Chú ý với 22 = `10110`: bit 1 đầu tiên và bit 1 cuối cùng **không** kề nhau vì có bit 1 ở giữa.

**Constraints:** `1 <= n <= 10^9` (tức tối đa 30 bit).

In [1]:
for n in [22, 8, 5]:
    print(n, "->", bin(n), "->", bin(n)[2:])

22 -> 0b10110 -> 10110
8 -> 0b1000 -> 1000
5 -> 0b101 -> 101


## 2. Nền tảng Python cần dùng

### 2.1. `bin(n)` và vị trí bit

`bin(22)` trả chuỗi `'0b10110'` — tiền tố `'0b'` báo hiệu nhị phân; `bin(22)[2:]` bỏ tiền tố. Bit **vị trí 0** là bit phải nhất (giá trị `2^0 = 1`), vị trí `k` có giá trị `2^k`.

`22 = 16 + 4 + 2 = 2^4 + 2^2 + 2^1` ⇒ bit 1 ở các vị trí `4, 2, 1`.

In [2]:
n = 22
s = bin(n)[2:]
print(s)
for k in range(len(s)):
    bit = s[len(s) - 1 - k]          # đọc từ phải sang trái
    print(f"vị trí {k}: bit {bit}  (giá trị {int(bit) * 2**k})")

10110
vị trí 0: bit 0  (giá trị 0)
vị trí 1: bit 1  (giá trị 2)
vị trí 2: bit 1  (giá trị 4)
vị trí 3: bit 0  (giá trị 0)
vị trí 4: bit 1  (giá trị 16)


### 2.2. Toán tử bit: `&`, `>>`

- `n & 1`: AND từng bit với `1` ⇒ lấy **bit cuối** (0 hoặc 1) — giống `n % 2`.
- `n >> 1`: **dịch phải** 1 bit ⇒ bỏ bit cuối — giống `n // 2`.

Lặp hai thao tác này là cách "đọc" từng bit từ phải sang trái mà không cần chuỗi.

In [3]:
n = 22
while n > 0:
    print(f"n={n:>2} ({bin(n)[2:]:>5}) | n & 1 = {n & 1} | n >> 1 = {n >> 1}")
    n >>= 1

n=22 (10110) | n & 1 = 0 | n >> 1 = 11
n=11 ( 1011) | n & 1 = 1 | n >> 1 = 5
n= 5 (  101) | n & 1 = 1 | n >> 1 = 2
n= 2 (   10) | n & 1 = 0 | n >> 1 = 1
n= 1 (    1) | n & 1 = 1 | n >> 1 = 0


`n >>= 1` là viết tắt của `n = n >> 1`.

### 2.3. `enumerate` trên chuỗi đảo ngược

`enumerate(reversed(s))` cho cặp `(vị trí bit, ký tự)` theo đúng quy ước "bit 0 bên phải". (Thật ra để tính **khoảng cách** thì đọc từ trái hay phải đều như nhau — chỉ hiệu số mới quan trọng.)

In [4]:
s = "10110"
print([(k, ch) for k, ch in enumerate(reversed(s))])
print("vị trí các bit 1:", [k for k, ch in enumerate(reversed(s)) if ch == "1"])

[(0, '0'), (1, '1'), (2, '1'), (3, '0'), (4, '1')]
vị trí các bit 1: [1, 2, 4]


### 2.4. `max` với `default` và trên dãy rỗng

`max([])` báo lỗi; `max([], default=0)` trả `0`. Hữu ích khi không có cặp nào.

In [5]:
print(max([2, 1]), max([], default=0))
try:
    max([])
except ValueError as e:
    print("ValueError:", e)

2 0
ValueError: max() iterable argument is empty


## 3. Cách 1 — Chuỗi nhị phân + danh sách vị trí bit 1

**Quan sát:** các cặp "kề nhau" chính là các bit 1 **liên tiếp trong danh sách vị trí**. Nếu vị trí bit 1 là `p0 < p1 < ... < pk`, các cặp kề nhau là `(p0, p1), (p1, p2), ...` và đáp án là `max(p[i+1] - p[i])`.

In [6]:
def binary_gap_positions(n: int) -> int:
    s = bin(n)[2:]
    pos = [k for k, ch in enumerate(s) if ch == "1"]          # đọc trái -> phải cũng được
    return max((pos[i + 1] - pos[i] for i in range(len(pos) - 1)), default=0)

print(binary_gap_positions(22), binary_gap_positions(8), binary_gap_positions(5))

2 0 2


### Nhận xét

Đúng và gọn. Chỉ có `≤ 30` bit nên mọi cách đều nhanh; nhưng cách này dùng thêm chuỗi và list `O(log n)`. Hai bước tiếp theo sẽ:
1. bỏ list vị trí (chỉ nhớ vị trí bit 1 **gần nhất**),
2. bỏ luôn chuỗi (đọc bit bằng `&`, `>>`).

## 4. Cách 2 (brute force) — thử mọi cặp bit 1 và kiểm tra "kề nhau"

Để thấy rõ định nghĩa "kề nhau", viết cách thô: với mọi cặp vị trí `i < j` đều là bit 1, kiểm tra **không có bit 1 nào ở giữa**. `O(b³)` với `b` = số bit — chỉ để minh họa và làm oracle.

In [7]:
def binary_gap_brute(n: int) -> int:
    s = bin(n)[2:]
    best = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s)):
            if s[i] == "1" and s[j] == "1" and "1" not in s[i + 1:j]:
                best = max(best, j - i)
    return best

print(binary_gap_brute(22), binary_gap_brute(8), binary_gap_brute(5))

2 0 2


`"1" not in s[i+1:j]` kiểm tra đoạn ở giữa không có ký tự `'1'` — đúng nghĩa "chỉ có 0 ở giữa".

## 5. Insight — chỉ cần nhớ vị trí bit 1 **gần nhất**

Duyệt các bit theo thứ tự (phải → trái). Mỗi khi gặp bit 1 ở vị trí `k`:
- nếu đã gặp bit 1 trước đó ở vị trí `last`, thì `(last, k)` là một cặp **kề nhau** (vì mọi bit giữa chúng đã được duyệt và đều là 0 — nếu có bit 1 thì `last` đã được cập nhật thành nó);
- cập nhật `best = max(best, k - last)` rồi `last = k`.

Kết hợp với `n & 1` và `n >>= 1`, ta đọc bit mà không cần chuỗi.

Đây là cùng kỹ thuật "nhớ vị trí gần nhất" như ở bài 821 (Shortest Distance to a Character).

## 6. Lời giải chính — thao tác bit với `last` (từng đoạn rồi ghép)

**Đoạn 1:** `last = -1` (chưa gặp bit 1 nào), `best = 0`, `k = 0` (vị trí bit đang xét).

**Đoạn 2:** `while n > 0:` — còn bit để đọc.

**Đoạn 3:** nếu `n & 1 == 1`: nếu `last != -1` thì `best = max(best, k - last)`; sau đó `last = k`.

**Đoạn 4:** `n >>= 1; k += 1`.

In [8]:
def binary_gap_bits(n: int) -> int:
    last = -1
    best = 0
    k = 0
    while n > 0:
        if n & 1:
            if last != -1:
                best = max(best, k - last)
            last = k
        n >>= 1
        k += 1
    return best

print(binary_gap_bits(22), binary_gap_bits(8), binary_gap_bits(5))

2 0 2


## 7. TRACE từng bước

### 7.1. Bảng trace `n = 22` (`10110`)

| k | n (nhị phân) | n & 1 | last trước | khoảng cách | best | last sau |
|---|---|---|---|---|---|---|
| 0 | 10110 | 0 | −1 | — | 0 | −1 |
| 1 | 1011 | 1 | −1 | (bit 1 đầu tiên) | 0 | 1 |
| 2 | 101 | 1 | 1 | 2 − 1 = 1 | 1 | 2 |
| 3 | 10 | 0 | 2 | — | 1 | 2 |
| 4 | 1 | 1 | 2 | 4 − 2 = 2 | 2 | 4 |

`n` thành 0 ⇒ dừng. Kết quả `2`.

In [9]:
def trace_binary_gap(n: int) -> int:
    print(f"n = {n} = {bin(n)[2:]}")
    last, best, k = -1, 0, 0
    while n > 0:
        bit = n & 1
        note = ""
        if bit:
            if last != -1:
                best = max(best, k - last)
                note = f"cặp ({last},{k}) cách {k - last}"
            else:
                note = "bit 1 đầu tiên"
            last = k
        print(f"k={k} n={bin(n)[2:]:>6} bit={bit} last={last:>2} best={best} {note}")
        n >>= 1
        k += 1
    print("Kết quả:", best)
    return best

trace_binary_gap(22);

n = 22 = 10110
k=0 n= 10110 bit=0 last=-1 best=0 
k=1 n=  1011 bit=1 last= 1 best=0 bit 1 đầu tiên
k=2 n=   101 bit=1 last= 2 best=1 cặp (1,2) cách 1
k=3 n=    10 bit=0 last= 2 best=1 
k=4 n=     1 bit=1 last= 4 best=2 cặp (2,4) cách 2
Kết quả: 2


### 7.2. Trace lũy thừa của 2 (chỉ một bit 1) và số có khoảng trống lớn

In [10]:
trace_binary_gap(8);
print()
trace_binary_gap(0b1000101);

n = 8 = 1000
k=0 n=  1000 bit=0 last=-1 best=0 
k=1 n=   100 bit=0 last=-1 best=0 
k=2 n=    10 bit=0 last=-1 best=0 
k=3 n=     1 bit=1 last= 3 best=0 bit 1 đầu tiên
Kết quả: 0

n = 69 = 1000101
k=0 n=1000101 bit=1 last= 0 best=0 bit 1 đầu tiên
k=1 n=100010 bit=0 last= 0 best=0 
k=2 n= 10001 bit=1 last= 2 best=2 cặp (0,2) cách 2
k=3 n=  1000 bit=0 last= 2 best=2 
k=4 n=   100 bit=0 last= 2 best=2 
k=5 n=    10 bit=0 last= 2 best=2 
k=6 n=     1 bit=1 last= 6 best=4 cặp (2,6) cách 4
Kết quả: 4


## 8. Code nộp LeetCode

In [11]:
class Solution:
    def binaryGap(self, n: int) -> int:
        last = -1
        best = 0
        k = 0
        while n:
            if n & 1:
                if last >= 0:
                    best = max(best, k - last)
                last = k
            n >>= 1
            k += 1
        return best

sol = Solution()
assert sol.binaryGap(22) == 2
assert sol.binaryGap(8) == 0
assert sol.binaryGap(5) == 2
print("Solution PASS 3 ví dụ đề")

Solution PASS 3 ví dụ đề


Bản chuỗi một lượt (cũng nộp được, dễ đọc):

In [12]:
class SolutionString:
    def binaryGap(self, n: int) -> int:
        last = -1
        best = 0
        for k, ch in enumerate(bin(n)[2:]):
            if ch == "1":
                if last >= 0:
                    best = max(best, k - last)
                last = k
        return best

print(SolutionString().binaryGap(22), SolutionString().binaryGap(8), SolutionString().binaryGap(5))

2 0 2


## 9. Vì sao đúng? Độ phức tạp

**Invariant:** sau khi đọc các bit ở vị trí `0..k-1`:
- `last` là vị trí **lớn nhất** trong số đó có bit 1 (hoặc −1 nếu chưa có);
- `best` là khoảng cách lớn nhất giữa các cặp bit 1 kề nhau **nằm hoàn toàn** trong `0..k-1`.

Khi đọc bit `k`:
- bit 0: không tạo cặp mới, `last` không đổi — invariant giữ.
- bit 1: cặp kề nhau **mới** duy nhất có đầu phải là `k` là `(last, k)` — vì `last` là bit 1 gần nhất trong phần đã đọc, nên mọi bit giữa `last` và `k` đều là 0. Cập nhật `best` với `k - last` rồi `last = k` — invariant giữ.

Khi `n == 0`, mọi bit 1 đã được đọc (các bit cao hơn đều là 0) ⇒ `best` là đáp án.

**Độ phức tạp:** gọi `b = số bit của n ≈ log₂ n` (≤ 30).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Chuỗi + list vị trí | `O(b)` | `O(b)` |
| 2. Brute force mọi cặp | `O(b³)` | `O(b)` |
| 3. Chuỗi một lượt (`last`) | `O(b)` | `O(b)` (chuỗi `bin`) |
| 4. Bit thuần (`&`, `>>`) | `O(b)` = `O(log n)` | `O(1)` |

## 10. Cách 5 — Nhảy thẳng tới bit 1 tiếp theo: `n & -n` và `bit_length`

Mẹo bit nâng cao:
- `n & -n` cô lập **bit 1 thấp nhất** (ví dụ `22 & -22 = 2` = `0b10`).
- `x.bit_length() - 1` cho **vị trí** của bit đó.
- `n & (n - 1)` **xóa** bit 1 thấp nhất.

Vòng lặp chỉ chạy **số lần = số bit 1**, không phải số bit.

In [13]:
n = 22
print("n & -n =", n & -n, "| vị trí:", (n & -n).bit_length() - 1, "| n & (n-1) =", bin(n & (n - 1)))

def binary_gap_lowbit(n: int) -> int:
    last = -1
    best = 0
    while n:
        low = n & -n                        # bit 1 thấp nhất
        pos = low.bit_length() - 1          # vị trí của nó
        if last >= 0:
            best = max(best, pos - last)
        last = pos
        n &= n - 1                          # xóa bit đó
    return best

print(binary_gap_lowbit(22), binary_gap_lowbit(8), binary_gap_lowbit(5))

n & -n = 2 | vị trí: 1 | n & (n-1) = 0b10100
2 0 2


Vì sao `n & -n` cô lập bit thấp nhất? Trong biểu diễn bù 2, `-n = ~n + 1`: mọi bit từ bit 1 thấp nhất trở lên bị đảo, trừ chính bit đó. AND lại chỉ còn bit ấy. (Python số nguyên không giới hạn bit nhưng vẫn cư xử như bù 2 vô hạn cho các phép bit.)

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. `n = 1` (một bit 1) → 0; lũy thừa của 2 → 0.
3. Toàn bit 1 (`7 = 111`) → 1.
4. Bit 1 chỉ ở hai đầu (`2^29 + 1`) → 29.
5. Phản ví dụ cho ý tưởng sai "lấy khoảng cách bit 1 đầu và cuối" (`22`).
6. Giá trị lớn nhất `10^9`.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1 (22)", sol.binaryGap(22), 2)
check("đề 2 (8)", sol.binaryGap(8), 0)
check("đề 3 (5)", sol.binaryGap(5), 2)
check("n=1", sol.binaryGap(1), 0)
check("2^20", sol.binaryGap(2**20), 0)
check("toàn 1 (7)", sol.binaryGap(7), 1)
check("hai đầu", sol.binaryGap(2**29 + 1), 29)
check("6 = 110", sol.binaryGap(6), 1)
check("10^9", sol.binaryGap(10**9), binary_gap_brute(10**9))
print("10^9 =", bin(10**9)[2:])

PASS đề 1 (22): 2
PASS đề 2 (8): 0
PASS đề 3 (5): 2
PASS n=1: 0
PASS 2^20: 0
PASS toàn 1 (7): 1
PASS hai đầu: 29
PASS 6 = 110: 1
PASS 10^9: 3
10^9 = 111011100110101100101000000000


### 11.1. Đối chiếu 6 cài đặt trên mọi `n` từ 1 đến 5000 và mẫu ngẫu nhiên lớn (có seed)

In [15]:
import random
random.seed(868)
impls = [binary_gap_positions, binary_gap_bits, binary_gap_lowbit,
         Solution().binaryGap, SolutionString().binaryGap]
tests = list(range(1, 5001)) + [random.randint(1, 10**9) for _ in range(2000)]
for n in tests:
    expected = binary_gap_brute(n)
    for f in impls:
        assert f(n) == expected, (n, f)
print(f"{len(tests)} giá trị: 6 cài đặt khớp nhau")

7000 giá trị: 6 cài đặt khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: lấy khoảng cách giữa bit 1 **đầu tiên và cuối cùng**

Sai định nghĩa "kề nhau" — với 22 = `10110` ra 3 thay vì 2.

In [16]:
def wrong_first_last(n):
    s = bin(n)[2:]
    return s.rfind("1") - s.find("1")

print("sai :", wrong_first_last(22), "| đúng:", Solution().binaryGap(22))

sai : 3 | đúng: 2


`s.find(x)` trả chỉ số xuất hiện **đầu tiên**, `s.rfind(x)` trả chỉ số xuất hiện **cuối cùng** (`-1` nếu không có).

### Lỗi 2: đếm số bit 0 liên tiếp thay vì khoảng cách

Khoảng cách giữa hai bit 1 = (số bit 0 ở giữa) **+ 1**. `"101"` có một bit 0 nhưng khoảng cách là 2.

In [17]:
def wrong_count_zeros(n):
    s = bin(n)[2:].strip("0")             # bỏ 0 ở hai đầu
    return max((len(z) for z in s.split("1")), default=0)

print("sai :", wrong_count_zeros(5), "| đúng:", Solution().binaryGap(5))
print("sai :", wrong_count_zeros(7), "| đúng:", Solution().binaryGap(7), "<- còn sai ở ca toàn 1")

sai : 1 | đúng: 2
sai : 0 | đúng: 1 <- còn sai ở ca toàn 1


### Lỗi 3: quên chuỗi `bin` có tiền tố `'0b'`

Duyệt `bin(n)` mà không cắt `[2:]` — ký tự `'b'`, `'0'` đầu sẽ làm lệch vị trí (may mắn ở bài này vì chỉ lấy hiệu, nhưng ở bài đếm bit 0 thì sai ngay).

In [18]:
print(bin(5), "->", list(bin(5)), "| đúng:", list(bin(5)[2:]))
print("số bit 0 trong bin(5):", bin(5).count("0"), "<- sai (đếm cả '0' của '0b')", "| đúng:", bin(5)[2:].count("0"))

0b101 -> ['0', 'b', '1', '0', '1'] | đúng: ['1', '0', '1']
số bit 0 trong bin(5): 2 <- sai (đếm cả '0' của '0b') | đúng: 1


### Lỗi 4: khởi tạo `last = 0` thay vì `-1`

Khi bit 1 đầu tiên ở vị trí `k > 0`, code coi như có "bit 1 ảo" ở vị trí 0 ⇒ tạo cặp giả.

In [19]:
def wrong_last_init(n):
    last, best, k = 0, 0, 0              # SAI
    while n:
        if n & 1:
            best = max(best, k - last)
            last = k
        n >>= 1; k += 1
    return best

print("sai :", wrong_last_init(8), "| đúng:", Solution().binaryGap(8))

sai : 3 | đúng: 0


## 13. Mẹo nhận dạng pattern

- Đề nói "biểu diễn nhị phân", "bit" → hai lựa chọn: `bin(n)[2:]` (dễ đọc) hoặc vòng `n & 1` / `n >>= 1` (thuần bit, `O(1)` bộ nhớ).
- "Khoảng cách giữa các phần tử thỏa điều kiện **liên tiếp**" → nhớ **vị trí gần nhất** `last`, cập nhật khi gặp phần tử mới.
- Mẹo bit đáng nhớ: `n & 1` (bit cuối), `n >> 1` (bỏ bit cuối), `n & (n - 1)` (xóa bit 1 thấp nhất), `n & -n` (cô lập bit 1 thấp nhất), `n.bit_length()`.
- Bài liên quan: *Number of 1 Bits (191)*, *Power of Two (231)*, *Counting Bits (338)*, *Max Consecutive Ones (485)*, *Binary Number with Alternating Bits (693)*.

## 14. Biến thể

### 14.1. Khoảng 0 dài nhất **bị kẹp giữa hai bit 1** (Codility "BinaryGap")

Phiên bản nổi tiếng: đếm **số bit 0** dài nhất nằm giữa hai bit 1 = (đáp án 868) − 1, nhưng tối thiểu 0.

In [20]:
def longest_zero_run_between_ones(n: int) -> int:
    return max(0, Solution().binaryGap(n) - 1)

for n in [9, 529, 20, 15, 32]:
    print(n, bin(n)[2:], "->", longest_zero_run_between_ones(n))

9 1001 -> 2
529 1000010001 -> 4
20 10100 -> 1
15 1111 -> 0
32 100000 -> 0


### 14.2. Khoảng cách **nhỏ nhất** giữa hai bit 1 kề nhau (0 nếu chỉ có ≤ 1 bit 1)

In [21]:
def min_binary_gap(n: int) -> int:
    last, best, k = -1, None, 0
    while n:
        if n & 1:
            if last >= 0:
                best = k - last if best is None else min(best, k - last)
            last = k
        n >>= 1; k += 1
    return best or 0

print(min_binary_gap(22), min_binary_gap(0b1001001), min_binary_gap(8))

1 3 0


`best or 0`: nếu `best` là `None` (chưa có cặp nào) thì trả `0`.

### 14.3. Bit xen kẽ (LeetCode 693): mọi cặp bit kề nhau khác nhau

`n ^ (n >> 1)` sẽ toàn bit 1 nếu `n` xen kẽ. Kiểm tra "toàn bit 1": `x & (x + 1) == 0`.

In [22]:
def has_alternating_bits(n: int) -> bool:
    x = n ^ (n >> 1)
    return x & (x + 1) == 0

print([(n, bin(n)[2:], has_alternating_bits(n)) for n in [5, 7, 10, 11]])

[(5, '101', True), (7, '111', False), (10, '1010', True), (11, '1011', False)]


## 15. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Viết nhị phân của `n = 100`, liệt kê vị trí bit 1 và tính `binaryGap(100)`.

### Bài 2 — Dự đoán output
Không chạy, đoán: `13 & 1`, `13 >> 1`, `12 & -12`, `bin(6)[2:]`, `(1 << 5)`.

### Bài 3 — Điền TODO cho lời giải bit

In [23]:
def ex3_binary_gap(n: int) -> int:
    last, best, k = -1, 0, 0
    while n > 0:
        # TODO: nếu bit cuối là 1 -> cập nhật best (nếu last hợp lệ) và last
        # TODO: dịch n, tăng k
        pass
    return best

# assert ex3_binary_gap(22) == 2 and ex3_binary_gap(8) == 0 and ex3_binary_gap(5) == 2

### Bài 4 — Number of 1 Bits (LeetCode 191)
Đếm số bit 1 của `n` bằng `n & (n - 1)` (không dùng `bin`).

In [24]:
def ex4_hamming_weight(n: int) -> int:
    # TODO
    pass

# assert ex4_hamming_weight(11) == 3 and ex4_hamming_weight(128) == 1

### Bài 5 — Max Consecutive Ones trên bit
Tìm số bit 1 **liên tiếp** dài nhất trong biểu diễn nhị phân của `n`.

In [25]:
def ex5_max_consecutive_one_bits(n: int) -> int:
    # TODO
    pass

# assert ex5_max_consecutive_one_bits(0b1101110) == 3
# assert ex5_max_consecutive_one_bits(8) == 1

### Bài 6 — Mini challenge: Binary Gap trên mảng
Cho list `nums` gồm 0/1 (có ít nhất một số 1). Tìm khoảng cách lớn nhất giữa hai số 1 kề nhau (giống 868 nhưng trên mảng). Dùng lại kỹ thuật `last`.

In [26]:
def ex6_array_gap(nums: list[int]) -> int:
    # TODO
    pass

# assert ex6_array_gap([1, 0, 0, 1, 1, 0, 1]) == 3
# assert ex6_array_gap([0, 1, 0]) == 0

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: `100 = 64 + 32 + 4`.
- Bài 4: mỗi lần `n &= n - 1` xóa đúng một bit 1.
- Bài 5: đây là bài "run dài nhất" (giống 830) trên các bit.

**Mức 2 — cú pháp**
- Bài 3: `if n & 1: ...`; `n >>= 1; k += 1`.
- Bài 5: biến `cur` đếm run hiện tại: bit 1 → `cur += 1`, bit 0 → `cur = 0`; `best = max(best, cur)`.

**Mức 3 — gần code**
- Bài 4: `cnt = 0; while n: n &= n - 1; cnt += 1`.
- Bài 6: `for i, x in enumerate(nums): if x == 1: if last >= 0: best = max(best, i - last); last = i`.

## 17. Đáp án tham khảo

**Bài 1:** `100 = 1100100` (bit 1 ở vị trí 6, 5, 2 tính từ phải). Cặp kề nhau: (5, 6) cách 1; (2, 5) cách 3 ⇒ `3`.

**Bài 2:** `1`, `6`, `4`, `'110'`, `32`.

In [27]:
print(bin(100)[2:], 13 & 1, 13 >> 1, 12 & -12, bin(6)[2:], 1 << 5)
assert Solution().binaryGap(100) == 3

def ans3_binary_gap(n):
    last, best, k = -1, 0, 0
    while n > 0:
        if n & 1:
            if last != -1:
                best = max(best, k - last)
            last = k
        n >>= 1
        k += 1
    return best

def ans4_hamming_weight(n):
    cnt = 0
    while n:
        n &= n - 1
        cnt += 1
    return cnt

def ans5_max_consecutive_one_bits(n):
    best = cur = 0
    while n:
        if n & 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
        n >>= 1
    return best

def ans6_array_gap(nums):
    last, best = -1, 0
    for i, x in enumerate(nums):
        if x == 1:
            if last >= 0:
                best = max(best, i - last)
            last = i
    return best

assert ans3_binary_gap(22) == 2 and ans3_binary_gap(8) == 0 and ans3_binary_gap(5) == 2
assert ans4_hamming_weight(11) == 3 and ans4_hamming_weight(128) == 1
assert ans5_max_consecutive_one_bits(0b1101110) == 3
assert ans5_max_consecutive_one_bits(8) == 1
assert ans6_array_gap([1, 0, 0, 1, 1, 0, 1]) == 3
assert ans6_array_gap([0, 1, 0]) == 0
for n in range(1, 2000):
    assert ans6_array_gap([int(c) for c in bin(n)[2:]]) == Solution().binaryGap(n)
print("Đáp án bài 1-6 PASS (bài 6 khớp binaryGap trên 1..1999)")

1100100 1 6 4 110 32
Đáp án bài 1-6 PASS (bài 6 khớp binaryGap trên 1..1999)


## 18. Tổng kết — template nhớ nhanh và checklist submit

```python
last, best, k = -1, 0, 0
while n:
    if n & 1:                       # bit hiện tại là 1
        if last >= 0:
            best = max(best, k - last)
        last = k
    n >>= 1                         # sang bit kế tiếp
    k += 1
return best
```

**Câu thần chú:** *"Đọc bit bằng `& 1` và `>> 1`; nhớ vị trí bit 1 gần nhất; khoảng cách = hiệu vị trí."*

**Checklist trước khi submit:**
- [ ] "Kề nhau" = bit 1 **liên tiếp** trong danh sách vị trí, không phải đầu–cuối?
- [ ] Khoảng cách = hiệu vị trí (= số 0 ở giữa **+ 1**)?
- [ ] `last` khởi tạo `-1` (chưa có bit 1)?
- [ ] Nếu dùng `bin`, đã cắt `'0b'`?
- [ ] Trả `0` khi chỉ có một bit 1 (lũy thừa của 2)?